In [0]:
%run ../../config/utils

In [0]:
import datetime
import yaml
import sys
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
import pandas as pd

In [0]:
with open('./config/config.yml', "r") as f:
    cfg = yaml.load(f, Loader=yaml.FullLoader)

p = cfg["params"]

today = datetime.datetime.today()
recent_saturday = today - pd.offsets.Week(weekday=5)
recent_saturday_str = recent_saturday.strftime('%Y-%m-%d')

In [0]:
try:
    cube = (
        spark.table(fs_customer_cube_full)
        .filter(F.col("FISCAL_WEEK_END") == recent_saturday_str)
        .filter(F.col("MBRSHP_STAT_CD") == p["membership_stat"])
        .filter(F.col("LATEST_MBRSHP_EXP_DT") >= F.current_date() - 95)
        .filter(F.col("LATEST_MFI_TIER") > 0)
        .filter(F.col("TEAM_MBR_IND") != 'Y')
    )
    print("cube data loaded")
except Exception as e:
    print("error loading cube data", e)
    sys.exit(1)

In [0]:
cube = cube.withColumn("target_strt_dt", F.date_add("FISCAL_WEEK_END", 1)) \
           .withColumn("target_end_dt_1", F.date_add("FISCAL_WEEK_END", 7)) \
           .withColumn('target_end_dt_2', F.date_add("FISCAL_WEEK_END", 14))\
           .withColumn('target_end_dt_3', F.date_add("FISCAL_WEEK_END", 21))\
           .withColumn("last_7days", F.date_add("FISCAL_WEEK_END", -7+1)) \
           .withColumn("last_60days", F.date_add("FISCAL_WEEK_END", -60+1))\
           .withColumn('last_90days', F.date_add(F.col("FISCAL_WEEK_END"), -90+1))\
           .withColumn('last_180days', F.date_add(F.col("FISCAL_WEEK_END"), -180+1))



print(f"Member count: {cube.select('MBRSHP_SID').count()}")


hdr = spark.table(silver_transaction_fiscal_header).filter(F.col("PURCH_DT") >= p["min_purchase_date"]) \
             .filter(F.col("SALES_CHANNEL_ID").isin(p["sales_channels"])).select('MBRSHP_SID','PURCH_HDR_ID','SALES_CHANNEL_ID')



dtl = spark.table(silver_transaction_fiscal_detail).filter(F.col("PURCH_DT") >= p["min_purchase_date"]) \
             .filter(F.col("SALES_CTGRY_CD").isin(p["merchandise_ctgry"])) \
             .filter(F.col('RETURN_IND')=='N')\
             .select('PURCH_DT','PURCH_HDR_ID','DISCOUNT_TYPE_CD','ARTICLE_NBR','EXTENDED_PRC_AMT','SALES_CTGRY_CD','MCH3_CD','MCH2_CD','MCH2_DESC','MCH1_CD','AH3_DESC','AH4_DESC')



pay = spark.table(silver_transaction_fiscal_payment).filter(F.col("PURCH_DT") >= p["min_purchase_date"])\
.select('MBRSHP_SID','PURCH_HDR_ID','PURCH_DT','TENDER_TYPE_CD','SALES_PYMT_AMT')



purch = dtl.join(hdr, ["PURCH_HDR_ID"], how="left")


sales_data = cube.join(purch,'MBRSHP_SID','left').fillna(0,subset=['EXTENDED_PRC_AMT'])



sales_data_v2 = sales_data.groupBy("MBRSHP_SID", "FISCAL_WEEK_END").agg(
    # Target Spend
    F.sum(F.when((F.col('PURCH_DT').between(F.col('target_strt_dt'), F.col('target_end_dt_1')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &(F.col('MCH3_CD').isin(p["mch3_codes"])) &
    F.col('DISCOUNT_TYPE_CD').isNull()),F.col('EXTENDED_PRC_AMT'))).alias('target1_spend'),

    F.sum(F.when((F.col('PURCH_DT').between(F.col('target_strt_dt'), F.col('target_end_dt_2')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &(F.col('MCH3_CD').isin(p["mch3_codes"])) &
    F.col('DISCOUNT_TYPE_CD').isNull()),F.col('EXTENDED_PRC_AMT'))).alias('target2_spend'),

    F.sum(F.when((F.col('PURCH_DT').between(F.col('target_strt_dt'), F.col('target_end_dt_3')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &(F.col('MCH3_CD').isin(p["mch3_codes"])) &
    F.col('DISCOUNT_TYPE_CD').isNull()),F.col('EXTENDED_PRC_AMT'))).alias('target3_spend'),

    # Last Purchase
    F.max(F.when((F.col('PURCH_DT') <= F.col('FISCAL_WEEK_END')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &(F.col('EXTENDED_PRC_AMT') > 0) &
    F.col('DISCOUNT_TYPE_CD').isNull(),F.col('PURCH_DT'))).alias('Last_Purchase'),

    # Sales
    F.sum(F.when((F.col('PURCH_DT').between(F.col('last_7days'), F.col('FISCAL_WEEK_END')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &(F.col('MCH3_CD').isin(p["mch3_codes"])) &
    F.col('DISCOUNT_TYPE_CD').isNull()),F.col('EXTENDED_PRC_AMT'))).alias('last_7days_sales'),

    F.sum(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) & (F.col('MCH3_CD').isin(p["mch3_codes"])) &
    F.col('DISCOUNT_TYPE_CD').isNull()),F.col('EXTENDED_PRC_AMT'))).alias('last_60days_sales'),

    # Trips
    F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_7days'), F.col('FISCAL_WEEK_END')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &
    (F.col('MCH3_CD').isin(p["mch3_codes"])) &(F.col('EXTENDED_PRC_AMT') > 0) & F.col('DISCOUNT_TYPE_CD').isNull()),F.col('PURCH_HDR_ID'))).alias('last_7days_trips'),

    F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &
    (F.col('MCH3_CD').isin(p["mch3_codes"])) & (F.col('EXTENDED_PRC_AMT') > 0) & F.col('DISCOUNT_TYPE_CD').isNull()), F.col('PURCH_HDR_ID'))).alias('last_60days_trips'),

    # Day trips
    F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END')) &(F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &
    (F.col('MCH3_CD').isin(p["mch3_codes"])) &(F.col('EXTENDED_PRC_AMT') > 0) & F.col('DISCOUNT_TYPE_CD').isNull()),F.col('PURCH_DT'))).alias('last_60days_daytrips'),

    # In-store
    F.sum(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END')) &(F.col('SALES_CHANNEL_ID') == p["sales_channels"][0]) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &
    (F.col('MCH3_CD').isin(p["mch3_codes"])) & F.col('DISCOUNT_TYPE_CD').isNull()),F.col('EXTENDED_PRC_AMT'))).alias('last_60days_instoresales'),

    F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END')) &(F.col('SALES_CHANNEL_ID') == p["sales_channels"][0]) &
    (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &(F.col('MCH3_CD').isin(p["mch3_codes"])) &(F.col('EXTENDED_PRC_AMT') > 0) & F.col('DISCOUNT_TYPE_CD').isNull()),
    F.col('PURCH_HDR_ID'))).alias('last_60days_instoretrips'),

    F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END')) & (F.col('SALES_CHANNEL_ID') == p["sales_channels"][0]) &
    (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) & (F.col('MCH3_CD').isin(p["mch3_codes"])) & (F.col('EXTENDED_PRC_AMT') > 0) & F.col('DISCOUNT_TYPE_CD').isNull()),
    F.col('PURCH_DT'))).alias('last_60days_instoredaytrips'),

    # Online - grocery
    F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END')) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &
    (F.col('MCH3_CD') == p["mch3_codes"][0]) & (F.col('EXTENDED_PRC_AMT') > 0) & F.col('DISCOUNT_TYPE_CD').isNull()),F.col('PURCH_HDR_ID'))).alias('last_60days_grocery_trips'),

    # Online - perishables
    F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END')) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) &
    (F.col('MCH3_CD') == p["mch3_codes"][1]) & (F.col('EXTENDED_PRC_AMT') > 0) & F.col('DISCOUNT_TYPE_CD').isNull()),F.col('PURCH_HDR_ID'))).alias('last_60days_perishables_trips')
)


# Calculate Basket size and % trips
sales_data_v2 = sales_data_v2.withColumn("LAST_7DAYS_BASKETSIZE", F.when(F.col("last_7days_trips").isNotNull(),F.coalesce(F.try_divide(F.col("LAST_7DAYS_SALES"),F.col("last_7days_trips")), F.lit(0))).otherwise(0))\
.withColumn("LAST_60DAYS_INSTORE_BASKETSIZE", F.when(F.col("last_60days_trips").isNotNull(), F.coalesce(F.try_divide(F.col("LAST_60DAYS_instoresales"),F.col("LAST_60DAYS_instoretrips")),F.lit(0))).otherwise(0))



sales_data_v2 = sales_data_v2.fillna(0)


# ## Avg Days between purchase


# ## Prepare the dataset only on merchandise sales
sales_data_days_btw_purchase = cube.join(purch,'MBRSHP_SID','left').filter((F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0])
                                                                                 & (F.col('MCH3_CD').isin(p["mch3_codes"]))
                                                                                 & (F.col('EXTENDED_PRC_AMT') >0)
                                                                                 & (F.col('DISCOUNT_TYPE_CD').isNull())).select('MBRSHP_SID','FISCAL_WEEK_END','PURCH_DT','last_60days','last_180days',).distinct()

# # Define a window to partition by customer and order by purchase date
window_spec = Window.partitionBy("MBRSHP_SID","FISCAL_WEEK_END").orderBy("PURCH_DT")

# # Calculate the difference in days between purchases
sales_data_days_btw_purchase = sales_data_days_btw_purchase.withColumn("previous_purchase_date", F.lag("PURCH_DT").over(window_spec))
sales_data_days_btw_purchase = sales_data_days_btw_purchase.withColumn("days_between", F.datediff(sales_data_days_btw_purchase["PURCH_DT"], sales_data_days_btw_purchase["previous_purchase_date"]))


average_days_df = sales_data_days_btw_purchase.groupBy("MBRSHP_SID","FISCAL_WEEK_END").agg(
     F.avg(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END'))) & (F.col('previous_purchase_date').between(F.col('last_60days'), F.col('FISCAL_WEEK_END'))),F.col('days_between'))).alias('average_days_between_last_60days'),
     F.avg(F.when((F.col('PURCH_DT').between(F.col('last_180days'), F.col('FISCAL_WEEK_END'))) & (F.col('previous_purchase_date').between(F.col('last_180days'), F.col('FISCAL_WEEK_END'))),F.col('days_between'))).alias('average_days_between_last_180days'))
    

# #Fill missing values by highest window gap
average_days_df = average_days_df.fillna(60,subset = ['average_days_between_last_60days'])
average_days_df = average_days_df.fillna(180,subset = ['average_days_between_last_180days'])

#average_days_df.cache()

# # Show the result
average_days_df.select('MBRSHP_SID','FISCAL_WEEK_END',
                       'average_days_between_last_60days',
                       'average_days_between_last_180days',)



## Change over quarter
# sales metrics
quarter_change = cube.join(purch,'MBRSHP_SID','left').fillna(0,subset=['EXTENDED_PRC_AMT'])

quarter_change_v2 = quarter_change.groupby('MBRSHP_SID','FISCAL_WEEK_END').agg(
                                                       F.sum(F.when((F.col('PURCH_DT').between(F.col('last_180days'), F.date_add(F.col('last_90days'),1))) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) & (F.col('MCH3_CD').isin(p["mch3_codes"])) & (F.col('DISCOUNT_TYPE_CD').isNull()), F.col('EXTENDED_PRC_AMT'))).alias('before_last_quarter_sales'), 
                                                       F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_180days'), F.date_add(F.col('last_90days'),1))) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) & (F.col('MCH3_CD').isin(p["mch3_codes"])) & (F.col('DISCOUNT_TYPE_CD').isNull()), F.col('PURCH_DT'))).alias('before_last_quarter_daytrips'),)

quarter_change_v2 = quarter_change_v2.withColumn("before_last_quarter_basketsize", F.when(F.col("before_last_quarter_daytrips").isNotNull(), F.coalesce(F.try_divide(F.col("before_last_quarter_sales"),F.col("before_last_quarter_daytrips")), F.lit(0))).otherwise(0))\


quarter_change_v2 = quarter_change_v2.fillna(0)

#quarter_change_v2.show()

# ## Change over 60 days


## Change over 60days
# sales metrics
last60days_change = cube.join(purch,'MBRSHP_SID','left').fillna(0,subset=['EXTENDED_PRC_AMT'])

last60days_change_v2 = last60days_change.groupby('MBRSHP_SID','FISCAL_WEEK_END').agg(
                                                       #Quarter sales
                                                       F.sum(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END'))) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) & (F.col('MCH3_CD').isin(p["mch3_codes"])) & (F.col('DISCOUNT_TYPE_CD').isNull()), F.col('EXTENDED_PRC_AMT'))).alias('last_q60days_sales'), 
                                                       F.sum(F.when((F.col('PURCH_DT').between(F.date_add(F.col('last_60days'),-60), F.date_add(F.col('last_60days'),1))) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0])& (F.col('MCH3_CD').isin(p["mch3_codes"])) & (F.col('DISCOUNT_TYPE_CD').isNull()), F.col('EXTENDED_PRC_AMT'))).alias('before_last_q60days_sales'), 

                                                       #Quarter Trip (days)
                                                       F.countDistinct(F.when((F.col('PURCH_DT').between(F.col('last_60days'), F.col('FISCAL_WEEK_END'))) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0])& (F.col('MCH3_CD').isin(p["mch3_codes"]))& (F.col('DISCOUNT_TYPE_CD').isNull()), F.col('PURCH_DT'))).alias('last_q60days_daytrips'), 
                                                       F.countDistinct(F.when((F.col('PURCH_DT').between(F.date_add(F.col('last_60days'),-60), F.date_add(F.col('last_60days'),1))) & (F.col('SALES_CTGRY_CD') == p["merchandise_ctgry"][0]) & (F.col('MCH3_CD').isin(p["mch3_codes"])) & (F.col('DISCOUNT_TYPE_CD').isNull()), F.col('PURCH_DT'))).alias('before_last_q60days_daytrips'),)

last60days_change_v2 = last60days_change_v2.withColumn("last_q60days_basketsize", F.when(F.col("last_q60days_daytrips").isNotNull(),F.coalesce(F.try_divide(F.col("last_q60days_sales"),F.col("last_q60days_daytrips")),F.lit(0))).otherwise(0))\
 .withColumn("before_last_q60days_basketsize", F.when(F.col("before_last_q60days_daytrips").isNotNull(),F.coalesce(F.try_divide(F.col("before_last_q60days_sales"),F.col("before_last_q60days_daytrips")),F.lit(0))).otherwise(0))\

last60days_change_v2 = last60days_change_v2.withColumn("before_last_q60days_basketsize", F.when(F.col("before_last_q60days_daytrips").isNotNull(),F.coalesce(F.try_divide(F.col("before_last_q60days_sales"),F.col("before_last_q60days_daytrips")),F.lit(0))).otherwise(0))



last60days_change_v2 = last60days_change_v2.fillna(0)

In [0]:
print("Sales Data:",sales_data_v2.count())
print("Members in Sales Data:", sales_data_v2.select('MBRSHP_SID').count())
print("Members per week in Sales Data:", sales_data_v2.select('MBRSHP_SID','FISCAL_WEEK_END').count())
print("Unique Members in Sales Data:", sales_data_v2.select('MBRSHP_SID').distinct().count())
print("Unique Members per week in Sales Data:", sales_data_v2.select('MBRSHP_SID','FISCAL_WEEK_END').distinct().count())
print("Quarter Change Data Count:",quarter_change_v2.count())
print("Average Days Data Count:", average_days_df.count())

In [0]:
try:
    sales_data_v3 = (sales_data_v2.join(quarter_change_v2, ['MBRSHP_SID', 'FISCAL_WEEK_END'], 'left').join(last60days_change_v2, ['MBRSHP_SID', 'FISCAL_WEEK_END'], 'left')
    .join(average_days_df, ['MBRSHP_SID', 'FISCAL_WEEK_END'], 'left'))
    print("successully joined sales_data_v3")
except Exception as e :
    print("Error while joining data into sales_data_v3:", e)
    sys.exit(1)


# Convert columns to a list of column names
columns = sales_data_v3.columns

# Check for duplicate columns
duplicate_columns = [col for col in columns if columns.count(col) > 1]

# Show the result
print("Duplicate column names:", set(duplicate_columns))

In [0]:
base_Data = sales_data_v3

print("base data count :", base_Data.count())

base_Data = sales_data_v3.join(cube,["MBRSHP_SID",'FISCAL_WEEK_END'],'inner')


# ## Features for Trip and Sales model


base_Data_v2 = base_Data.select(
    
  #Trips Model
'AVERAGE_DAYS_BETWEEN_LAST_180DAYS',
 'BEFORE_LAST_Q60DAYS_DAYTRIPS',
 'AVERAGE_DAYS_BETWEEN_LAST_60DAYS',
 'LAST_60DAYS_PERISHABLES_TRIPS',
 'LAST_60DAYS_GROCERY_TRIPS',
 'LAST_TWENTY-SIX_WEEK_DISTINCT_DAYS',

    
  #Sales Model                               
'BEFORE_LAST_Q60DAYS_BASKETSIZE',
 'L52W_MEDIAN_BASKETSIZE',
 'BEFORE_LAST_Q60DAYS_SALES',
 'BEFORE_LAST_QUARTER_BASKETSIZE',
 'BEFORE_LAST_QUARTER_SALES',
 'L52W_G4W_STDEV_SPEND',
 'LAST_60DAYS_INSTORE_BASKETSIZE',
 'LAST_60DAYS_SALES',

'LATEST_MBRSHP_NBR',                                
'MBRSHP_SID',
'FISCAL_WEEK_END')


base_Data_v2.dtypes

base_Data_v2 = base_Data_v2.fillna({ 
    'AVERAGE_DAYS_BETWEEN_LAST_60DAYS': 60,
    'AVERAGE_DAYS_BETWEEN_LAST_180DAYS': 180
    })


In [0]:
base_Data_v2.write.mode("overwrite").option('replaceWhere', f"FISCAL_WEEK_END = '{recent_saturday_str}'").saveAsTable(digital_propensity_features)